# Structure Probe Prediction Viewer

Interactive inspection of row/column/box candidate probe predictions on the current board state.


In [ ]:
from pathlib import Path
import os, sys

# Allow running from repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from sudoku.probes.session import ProbeSession
from sudoku.probes.modes import cell_candidates
from sudoku.probes.activations import build_grid_at_step
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN, END_CLUES_TOKEN, PAD_TOKEN_BT

In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
LAYER = 4     # layer to probe (0-indexed)
N_CELLS = 81    # probe all cells
SEED = 42

## Load session

In [ ]:
session = ProbeSession(CACHE_PATH)

## Classify puzzles: simple vs hard

Simple = no PUSH/POP tokens anywhere in the sequence (pure constraint propagation).  
Hard = at least one PUSH token (backtracking was used).

In [ ]:
is_simple = np.array([
    not any(t == PUSH_TOKEN for t in seq)
    for seq in session.sequences
])

n_simple = is_simple.sum()
n_hard = (~is_simple).sum()
print(f"Simple puzzles (no backtracking): {n_simple:,}")
print(f"Hard puzzles (with backtracking): {n_hard:,}")

# Sequence length distribution check
seq_lens = np.array([len(s) for s in session.sequences])
print(f"\nSimple seq len: min={seq_lens[is_simple].min()}, max={seq_lens[is_simple].max()}, mean={seq_lens[is_simple].mean():.1f}")
print(f"Hard   seq len: min={seq_lens[~is_simple].min()}, max={seq_lens[~is_simple].max()}, mean={seq_lens[~is_simple].mean():.1f}")

## Build train/test split at step 0

- **Train**: all puzzles at step 0 (80% split)
- **Test**: the remaining 20%, further divided into simple/hard subsets

In [ ]:
idx0 = session.index.at_step(1).first_per_puzzle()
train_mask, test_mask = session.split(idx0, test_size=0.2, seed=SEED)

train_idx = idx0[train_mask]
test_idx  = idx0[test_mask]

# Identify which test puzzles are simple vs hard
test_puzzle_ids = test_idx.puzzle_idx
test_is_simple = is_simple[test_puzzle_ids]

print(f"Train: {len(train_idx):,}  |  Test: {len(test_idx):,}")
print(f"Test simple: {test_is_simple.sum():,}  |  Test hard: {(~test_is_simple).sum():,}")

## Helper functions

In [ ]:
from scripts.analysis.probe_experiments import (
    build_candidate_targets,
    fit_multilabel_probe as fit_cell_probe,
    eval_multilabel_probe as eval_cell_probe,
)


## Train probes at step 0

One multi-label LR probe per cell (81 probes total), trained on the training split at step 0.

In [ ]:
# Activations at step 0 for train and test splits
X_train_all = session.acts(train_idx, layer=LAYER)  # (n_train, d_model)
X_test_all  = session.acts(test_idx, layer=LAYER)   # (n_test,  d_model)

grids_train = session.grids(train_idx)
grids_test  = session.grids(test_idx)

print(f"X_train: {X_train_all.shape}, X_test: {X_test_all.shape}")

In [ ]:
# Train one probe per cell — filter to cells that have empty instances in train
cell_probes = {}  # cell_idx -> (clf, train_empty_mask)

for cell_idx in tqdm(range(N_CELLS), desc="Training probes"):
    targets_train, is_empty_train = build_candidate_targets(grids_train, cell_idx)
    if is_empty_train.sum() < 10:
        continue  # skip cells with almost no empty instances in train
    clf = fit_cell_probe(X_train_all[is_empty_train], targets_train[is_empty_train])
    cell_probes[cell_idx] = clf

print(f"Trained probes for {len(cell_probes)} cells")

In [ ]:
# Pre-identify test puzzle IDs for quick lookup
test_puzzle_id_set = set(test_puzzle_ids.tolist())
simple_test_puzzle_ids = set(test_puzzle_ids[test_is_simple].tolist())
hard_test_puzzle_ids   = set(test_puzzle_ids[~test_is_simple].tolist())


In [ ]:
from scripts.analysis.probe_experiments import eval_multilabel_probe as eval_structure_probe


In [ ]:
from scripts.analysis.probe_experiments import build_structure_targets, fit_multilabel_probe

# --- Train one probe per substructure at step 0 ---
SUBTYPES = (
    [('row', i) for i in range(9)]
    + [('col', i) for i in range(9)]
    + [('box', i) for i in range(9)]
)

struct_probes = {}  # (subtype, sidx) -> clf
for subtype, sidx in tqdm(SUBTYPES, desc='Training structure probes'):
    y_tr = build_structure_targets(grids_train, subtype, sidx)
    struct_probes[(subtype, sidx)] = fit_multilabel_probe(X_train_all, y_tr)

print(f'Trained {len(struct_probes)} structure probes')


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

from sudoku.data_bt import (
    END_CLUES_TOKEN, PAD_TOKEN_BT, PUSH_TOKEN, POP_TOKEN, SUCCESS_TOKEN
)
from sudoku.probes.activations import build_grid_at_step


def _decode_token(tok):
    """Return human-readable string for a token."""
    if 0 <= tok <= 728:
        digit = (tok % 9) + 1
        col   = (tok // 9) % 9
        row   = tok // 81
        return f"(col={col+1}, row={row+1}, digit={digit})"
    elif tok == END_CLUES_TOKEN: return "<end_clues>"
    elif tok == PAD_TOKEN_BT:    return "<pad>"
    elif tok == PUSH_TOKEN:      return "<push>"
    elif tok == POP_TOKEN:       return "<pop>"
    elif tok == SUCCESS_TOKEN:   return "<success>"
    else:                        return f"<unknown:{tok}>"


def _get_seq_len(session, puzzle_id):
    seq = session.sequences[puzzle_id]
    for i, tok in enumerate(seq):
        if tok == PAD_TOKEN_BT:
            return i
    return len(seq)


def _struct_true_candidates(grid_str, subtype, sidx):
    """Boolean (9,): True if digit d+1 is still a candidate in this substructure."""
    candidates = np.ones(9, dtype=bool)
    if subtype == "row":
        cells = grid_str[sidx * 9:(sidx + 1) * 9]
    elif subtype == "col":
        cells = [grid_str[r * 9 + sidx] for r in range(9)]
    else:  # box
        br, bc = (sidx // 3) * 3, (sidx % 3) * 3
        cells = [grid_str[(br + dr) * 9 + (bc + dc)] for dr in range(3) for dc in range(3)]
    for ch in cells:
        if ch in "123456789":
            candidates[int(ch) - 1] = False
    return candidates


def _struct_proba(struct_probes, subtype, sidx, X):
    clf = struct_probes.get((subtype, sidx))
    if clf is None:
        return np.full(9, float("nan"))
    
    # MODIFIED
    proba_list = clf.predict_proba(X)
    
    
    
    return np.array([p[0, 1] for p in proba_list])


def _cell(ax, x0, y0, prob, is_cand, label, cmap):
    ax.add_patch(plt.Rectangle((x0, y0), 1, 1, color=cmap(prob), zorder=1))
    ax.text(x0 + 0.5, y0 + 0.63, label,
            ha="center", va="center", fontsize=7, zorder=2,
            fontweight="bold" if is_cand else "normal",
            color="#111" if is_cand else "#888")
    ax.text(x0 + 0.5, y0 + 0.24, f"{prob:.2f}",
            ha="center", va="center", fontsize=5, zorder=2, color="#111")


def _grid_lines(ax, w, h, thick_every=3):
    for i in range(h + 1):
        lw = 1.8 if i % thick_every == 0 else 0.4
        ax.plot([0, w], [i, i], color="black", linewidth=lw, zorder=3)
    for i in range(w + 1):
        lw = 1.8 if i % thick_every == 0 else 0.4
        ax.plot([i, i], [0, h], color="black", linewidth=lw, zorder=3)


def _draw_board(ax, grid_str):
    ax.set_xlim(0, 9); ax.set_ylim(0, 9)
    ax.set_aspect("equal"); ax.axis("off")
    for cell_idx in range(81):
        r, c = divmod(cell_idx, 9)
        y0   = 8 - r
        ch   = grid_str[cell_idx]
        ax.add_patch(plt.Rectangle((c, y0), 1, 1,
                                   color="#d8d8d8" if ch != "0" else "white", zorder=1))
        if ch != "0":
            ax.text(c + 0.5, y0 + 0.5, ch, ha="center", va="center",
                    fontsize=16, fontweight="bold", zorder=2)
    _grid_lines(ax, 9, 9)


def _draw_row_grid(ax, grid_str, struct_probes, X):
    """9 rows × 9 digit-columns. Row index on left, digit header on top."""
    cmap = plt.cm.RdYlGn
    ax.set_xlim(-0.55, 9.05); ax.set_ylim(-0.1, 9.55)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Row candidates + probe score", fontsize=8, pad=3)

    for d in range(9):
        ax.text(d + 0.5, 9.35, str(d + 1), ha="center", va="center", fontsize=6.5, color="#444")

    for sidx in range(9):
        true_cands = _struct_true_candidates(grid_str, "row", sidx)
        probas     = _struct_proba(struct_probes, "row", sidx, X)
        y0 = 8 - sidx
        ax.text(-0.2, y0 + 0.5, f"R{sidx+1}", ha="right", va="center", fontsize=6.5, color="#444")
        for d in range(9):
            _cell(ax, d, y0, float(probas[d]), bool(true_cands[d]), str(d + 1), cmap)

    _grid_lines(ax, 9, 9)


def _draw_col_grid(ax, grid_str, struct_probes, X):
    """9 column-strips across × 9 digit-rows down. Column header on top, digit on left."""
    cmap = plt.cm.RdYlGn
    ax.set_xlim(-0.1, 9.55); ax.set_ylim(-0.55, 9.05)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Column candidates + probe score", fontsize=8, pad=15)

    for sidx in range(9):
        ax.text(sidx + 0.5, 9.35, f"C{sidx+1}", ha="center", va="center", fontsize=6.5, color="#444")

    for d in range(9):
        ax.text(-0.2, 8 - d + 0.5, str(d + 1), ha="right", va="center", fontsize=6.5, color="#444")

    for sidx in range(9):          # x = column index
        true_cands = _struct_true_candidates(grid_str, "col", sidx)
        probas     = _struct_proba(struct_probes, "col", sidx, X)
        for d in range(9):         # y = digit (8-d so digit 1 is at top)
            _cell(ax, sidx, 8 - d, float(probas[d]), bool(true_cands[d]), str(d + 1), cmap)

    _grid_lines(ax, 9, 9)


def _draw_box_grid(ax, grid_str, struct_probes, X):
    """3×3 arrangement of boxes, each box contains a 3×3 digit sub-grid (digits 1–9)."""
    cmap = plt.cm.RdYlGn
    ax.set_xlim(-0.1, 9.55); ax.set_ylim(-0.1, 9.55)
    ax.set_aspect("equal"); ax.axis("off")
    ax.set_title("Box candidates + probe score", fontsize=8, pad=3)

    for sidx in range(9):          # box index 0..8, row-major (top-left first)
        br, bc = sidx // 3, sidx % 3
        true_cands = _struct_true_candidates(grid_str, "box", sidx)
        probas     = _struct_proba(struct_probes, "box", sidx, X)

        # Box origin in plot coords (y increases upward; br=0 → top → largest y)
        box_x0 = bc * 3
        box_y0 = (2 - br) * 3 


        for d in range(9):         # digits 1–9 arranged 3×3 within the box
            dc = d % 3             # digit column within box
            dr = d // 3            # digit row within box (0 = top)
            x0 = box_x0 + dc
            y0 = box_y0 + (2 - dr)
            _cell(ax, x0, y0, float(probas[d]), bool(true_cands[d]), str(d + 1), cmap)

    # Fine grid inside boxes (thin), thick box boundaries
    for i in range(10):
        lw = 2.0 if i % 3 == 0 else 0.4
        ax.plot([0, 9], [i, i], color="black", linewidth=lw, zorder=3)
        ax.plot([i, i], [0, 9], color="black", linewidth=lw, zorder=3)


def _show_puzzle_step(puzzle_id, seq_pos):
    seq        = session.sequences[puzzle_id]
    actual_len = _get_seq_len(session, puzzle_id)
    if seq_pos >= actual_len:
        print(f"seq_pos {seq_pos} >= sequence length {actual_len}")
        return

    tok        = int(seq[seq_pos])
    grid_str   = build_grid_at_step(seq, seq_pos)
    X          = session.activations[puzzle_id, LAYER, seq_pos][np.newaxis, :]

    fig, axes = plt.subplots(2, 2, figsize=(12, 9.5),
                              gridspec_kw={"hspace": 0.22, "wspace": 0.18}, dpi=300)
    fig.suptitle(
        f"Puzzle {puzzle_id} | seq_pos={seq_pos} | "
        f"token={tok}  | {_decode_token(tok)}",
        fontsize=10,
    )

    _draw_board(axes[0, 0], grid_str)
    axes[0, 0].set_title(f"Board state  (filled={81 - grid_str.count(chr(48))})",
                         fontsize=8, pad=3)

    _draw_row_grid(axes[0, 1], grid_str, struct_probes, X)

    _draw_col_grid(axes[1, 0], grid_str, struct_probes, X)
    _draw_box_grid(axes[1, 1], grid_str, struct_probes, X)

    extent = axes[0,1].get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    fig.savefig('only_ax1.png', bbox_inches=extent, dpi=200)
    extent = axes[1,0].get_tightbbox(fig.canvas.get_renderer()).transformed(fig.dpi_scale_trans.inverted())
    fig.savefig('only_ax2.png', bbox_inches=extent, dpi=200)

    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, norm=plt.Normalize(0, 1))
    sm.set_array([])
    fig.colorbar(sm, ax=[axes[0, 1], axes[1, 0], axes[1, 1]],
                 fraction=0.03, pad=0.02, label="P(candidate)")
    plt.show()


# ---- Widgets ----
_all_puzzle_ids = sorted(set(int(p) for p in session.index.puzzle_idx))

_puzzle_dd = widgets.Dropdown(
    options=_all_puzzle_ids,
    value=_all_puzzle_ids[0],
    description="Puzzle ID:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)

_seq_pos_slider = widgets.IntSlider(
    value=0, min=0, max=_get_seq_len(session, _all_puzzle_ids[0]) - 1, step=1,
    description="Seq pos:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)

def _update_slider_range(change):
    pid  = change["new"]
    slen = _get_seq_len(session, pid)
    _seq_pos_slider.max = slen - 1
    if _seq_pos_slider.value >= slen:
        _seq_pos_slider.value = 0

_puzzle_dd.observe(_update_slider_range, names="value")

_out = widgets.interactive_output(
    _show_puzzle_step,
    {"puzzle_id": _puzzle_dd, "seq_pos": _seq_pos_slider},
)

display(widgets.VBox([_puzzle_dd, _seq_pos_slider, _out]))
